$X = [x_1, x_2, \dots, x_k]$: sequence of embeddings for a role in a conversation

Closure: $1 - \frac{||x_k - x_1||_2}{\sum_{i=2}^k ||x_i - x_{i-1}||_2}$

Radial Consistency: $1 - \frac{\sigma_r}{\mu_r}$

where $r_i = ||x_i - \bar{x}||_2$ (the distance of each point to the centroid).

Circularity = Closure * Radial Consistency

In [20]:
from pathlib import Path
import numpy as np
import pandas as pd

from src.metrics import _velocity, _distance, _directness

def _compute_trajectory_circularity(traj: np.ndarray) -> float:
    """
    Measures how circular a trajectory is in high-dimensional space.
    Returns a float between 0.0 (straight line) and 1.0 (perfect circle).
    """
    # Need at least 3 points to form a meaningful polygon/circle
    if traj.ndim != 2 or traj.shape[0] < 3:
        return 0.0

    # 1. Closure (Does the path return to the start?)
    steps = traj[1:] - traj[:-1]
    step_distances = np.linalg.norm(steps, axis=1)
    total_path_length = np.sum(step_distances)

    if total_path_length < 1e-9:
        return 0.0  # No movement

    net_displacement = np.linalg.norm(traj[-1] - traj[0])
    
    # 1.0 = perfectly closed loop, 0.0 = straight line
    closure = max(0.0, 1.0 - (net_displacement / total_path_length))

    # 2. Radial Consistency (Are points equidistant from the topic center?)
    centroid = np.mean(traj, axis=0)
    radii = np.linalg.norm(traj - centroid, axis=1)
    
    mean_radius = np.mean(radii)
    if mean_radius < 1e-9:
        return 0.0
        
    # Coefficient of variation of radii
    cv_radii = np.std(radii) / mean_radius
    
    # 1.0 = perfect circle (all radii equal), lower = irregular shape
    radial_consistency = max(0.0, 1.0 - cv_radii)

    # Final Metric
    return float(closure * radial_consistency)

def calculate_circularity(emb: np.ndarray) -> dict[str, float]:
    """
    Separates an alternating (user, assistant) conversation matrix 
    and calculates circularity for both.
    """
    # Array slicing: 
    # [start:stop:step] -> 0::2 gets evens (user), 1::2 gets odds (assistant)
    user_emb = emb[0::2, :]
    assistant_emb = emb[1::2, :]
    overall_emb = emb
    
    return {
        "user_circularity": _compute_trajectory_circularity(user_emb),
        "assistant_circularity": _compute_trajectory_circularity(assistant_emb),
        "overall_circularity": _compute_trajectory_circularity(overall_emb)
    }

In [21]:
def calculate_metrics(experiment_name: str) -> pd.DataFrame:
    experiment_dir = f"/Users/muberraozmen/Development/psycho-pass/experiments/datasetV1_1772362234/{experiment_name}"

    data = pd.read_parquet(Path(experiment_dir) / "embeddings.parquet")
    embeddings = np.array(data["embeddings"].tolist())
    conversations = data.sort_values(by="sequence").groupby(["conversation_id"], as_index=False).agg(
        {
            "index": list,
            "objective": "first",
            "executed_turns": "max",
            "outcome": "last",
            "role": list,
        }, 
    )

    metrics = []
    for _, row in conversations.iterrows():

        indices = row["index"]
        emb = embeddings[indices, :]
        m = {
            # "conversation_id": row["conversation_id"],
            # "objective": row["objective"],
            "executed_turns": row["executed_turns"],
            "outcome": row["outcome"]
        }
        m.update(_velocity(emb))
        m.update(_distance(emb))
        m.update(_directness(emb))
        m.update(calculate_circularity(emb))
        metrics.append(m)


    metrics = pd.DataFrame(metrics)

    return metrics.groupby("outcome").mean().transpose()


tfidf_experiment ="embeddingsV1_1773707874"
e5_experiment = "embeddingsV3_1773709117"

tfidf_metrics = calculate_metrics(tfidf_experiment)
e5_metrics = calculate_metrics(e5_experiment)


In [25]:
tfidf_metrics

outcome,failure,success
executed_turns,7.000000,2.582153
avg_velocity,1.087681,1.087803
max_velocity,1.337033,1.205512
step_variance,0.024110,0.012595
total_path_length,14.139856,4.558312
final_displacement,1.346023,1.231580
directness_index,0.095528,0.488931
user_circularity,0.722582,0.222812
assistant_circularity,0.682057,0.211238
overall_circularity,0.792813,0.465213


In [23]:
e5_metrics

outcome,failure,success
executed_turns,7.000000,2.582153
avg_velocity,0.457626,0.431890
max_velocity,0.621679,0.511251
step_variance,0.008912,0.004661
total_path_length,5.949140,1.844172
final_displacement,0.598752,0.511926
directness_index,0.101257,0.498755
user_circularity,0.634176,0.201726
assistant_circularity,0.651127,0.206861
overall_circularity,0.763606,0.449903
